# PhysAR-IDS -- NB18 (v6): Attackers on the TU Delft digital-substation dataset

**Purpose.** Run the four attackers of Paper 1 against the NB17 detector on held-out runs, to see whether the ordering (white-box > budgeted PPO, random search, grey-box PGD) holds on a third dataset.

**Run after NB17** (it reads `cpps_windows_v6.csv` from `nb17_out`). Needs `sb3-contrib` and `gymnasium`.

**Target.** XGBoost on all 51 NB17 features (30 electrical, 9 host, 12 packet), 3-class (normal / fault / attack), trained on runs 1-3 of every variant, attacked on the attack windows of runs 4-5 (every one of them, about 180 windows in 12 runs).

**Threat model.** A network attacker controls the packet statistics only (counts, bytes, GOOSE/SV/other counts, source/destination counts, inter-arrival statistics). Electrical and host features are never changed. Budget: L-infinity, eps training-window standard deviations per feature, eps in {0.1, 0.3, 1.0}. Perturbed packet features are clipped to the training range, so counts cannot go negative or leave the range seen in training. Evasion = share of attack windows labelled normal or fault, **including windows the clean model already misses** (reported separately as `clean`; expect it to be near 0.28).

**Attackers.** Grey-box PGD through an MLP surrogate trained on the target's hard labels; random search (10 random +/- eps vertices per window, the same 10 queries as PPO); budgeted RecurrentPPO (LSTM 64, 10 steps, hard-label oracle, trained on the attack windows of the training runs only); white-box greedy search over every controllable feature using the tree thresholds (a lower bound on evasion).

**Statistics.** Seeds [42, 7, 123, 2024, 99]; paired t-tests over seeds with Holm correction (they measure model variance on a fixed split); 95% intervals from a bootstrap over **runs**.

**Caveats to state.** The 12 packet features are not independent in reality (GOOSE + SV + other counts sum to the frame count); the attacker here perturbs them independently, so this is an upper bound on what a real packet-level attacker could do. The attack windows are few, so intervals are wide.

In [ ]:
import os, time, json, warnings
import numpy as np, pandas as pd, joblib
from scipy import stats
import xgboost as xgb, torch, torch.nn as nn
from sklearn.metrics import f1_score
warnings.filterwarnings('ignore')
PIPELINE_VERSION = 'v6'
SEEDS = [int(s) for s in os.environ.get('NB18_SEEDS', '42,7,123,2024,99').split(',')]
EPS_LIST = [0.1, 0.3, 1.0]          # per-feature L-inf budget in training-window standard deviations
PPO_EPS = [0.1]
PPO_TIMESTEPS = int(os.environ.get('NB18_PPO_STEPS', 50000)); PPO_MAX_STEPS = 10
N_QUERIES = PPO_MAX_STEPS; PGD_ITERS = 10; K_CAND = None; WB_ITERS = 25; N_BOOT = 2000
OUT = os.environ.get('NB18_OUT', r'C:\Users\user\Desktop\smartrid1\nb17_out'); os.chdir(OUT)
TRAIN_RUNS, TEST_RUNS = {1, 2, 3}, {4, 5}
XGB_P = dict(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.6, min_child_weight=3, reg_lambda=1.0, tree_method='hist', n_jobs=-1, verbosity=0)
DEV = 'cuda' if torch.cuda.is_available() and not os.environ.get('NB18_CPU') else 'cpu'
print('xgboost', xgb.__version__, '| torch', torch.__version__, DEV, '| seeds', SEEDS, '| dir', os.getcwd())


In [ ]:
D = pd.read_csv(f'cpps_windows_{PIPELINE_VERSION}.csv')
ELEC = [c for c in D.columns if c.startswith('e_')]; HOST = [c for c in D.columns if c.startswith('h_')]; PKT = [c for c in D.columns if c.startswith('p_')]
FEATS = ELEC + HOST + PKT; F = len(FEATS)                      # same 'all' feature set as NB17
Xv = D[FEATS].values.astype('float32'); y3 = D.y.values.astype(int)
tr = np.flatnonzero(D.run.isin(TRAIN_RUNS).values); te = np.flatnonzero(D.run.isin(TEST_RUNS).values)
mu = Xv[tr].mean(0); sd = Xv[tr].std(0); sd1 = np.where(sd > 1e-9, sd, 1.0).astype('float32')
LO, HI = Xv[tr].min(0), Xv[tr].max(0)
PERT = np.array([i for i, f in enumerate(FEATS) if f.startswith('p_') and sd[i] > 1e-9])           # a network attacker controls packet statistics only
MASK = np.zeros(F, bool); MASK[PERT] = True
EV = te[y3[te] == 2]                                                                              # every attack window in the held-out runs
run_key = (D.variant.astype(str) + '_' + D.run.astype(str)).values
print(f'{len(tr)} train / {len(te)} test windows | {F} features | {len(PERT)} attacker-controlled packet features | {len(EV)} held-out attack windows in {len(np.unique(run_key[EV]))} runs')
print('controllable:', [FEATS[i] for i in PERT])


In [ ]:
def fit_target(seed): return xgb.XGBClassifier(**XGB_P, random_state=seed, num_class=3, objective='multi:softprob').fit(Xv[tr], y3[tr])
def proba(m, A): return m.predict_proba(np.ascontiguousarray(A, dtype=np.float32))
def evaded(m, A): return proba(m, A).argmax(1) != 2
def clip_pert(x):                                                   # perturbed packet statistics stay inside the training range (counts cannot go negative)
    x = x.copy(); x[:, PERT] = np.clip(x[:, PERT], LO[PERT], HI[PERT]); return x
def to_raw(z): return z * sd1 + mu
def to_z(A): return (A - mu) / sd1
class Surrogate(nn.Module):
    def __init__(s): super().__init__(); s.n = nn.Sequential(nn.Linear(F, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 3))
    def forward(s, x): return s.n(x)
def train_surrogate(target, seed, epochs=60):
    torch.manual_seed(seed); net = Surrogate().to(DEV); opt = torch.optim.Adam(net.parameters(), 1e-3)
    Z = torch.tensor(np.clip(to_z(Xv[tr]), -10, 10), dtype=torch.float32, device=DEV); y = torch.tensor(target.predict(Xv[tr]), dtype=torch.long, device=DEV)   # hard labels only
    for ep in range(epochs):
        perm = torch.randperm(len(Z), device=DEV)
        for i in range(0, len(Z), 128):
            b = perm[i:i + 128]; opt.zero_grad(); nn.functional.cross_entropy(net(Z[b]), y[b]).backward(); opt.step()
    return net.eval(), (net(Z).argmax(1) == y).float().mean().item()
def atk_pgd(net, rows, eps):
    z0 = torch.tensor(np.clip(to_z(Xv[rows]), -10, 10), dtype=torch.float32, device=DEV); m = torch.tensor(MASK, device=DEV).float(); z = z0.clone(); alpha = eps / 4
    for _ in range(PGD_ITERS):
        z.requires_grad_(True); lg = net(z); loss = torch.logsumexp(lg[:, [0, 1]], 1) - lg[:, 2]
        g, = torch.autograd.grad(loss.sum(), z); z = z.detach() + alpha * g.sign() * m; z = z0 + (z - z0).clamp(-eps, eps)
    delta = (z.detach() - z0).cpu().numpy().astype('float32'); return clip_pert((Xv[rows] + delta * sd1 * MASK).astype('float32'))
def atk_random(target, rows, eps, seed):
    rng = np.random.RandomState(seed); base = Xv[rows]; ok = evaded(target, base); out = base.copy()
    for k in range(N_QUERIES):
        todo = np.flatnonzero(~ok)
        if not len(todo): break
        cand = base[todo].copy(); cand[:, PERT] += eps * sd1[PERT] * rng.choice([-1.0, 1.0], size=(len(todo), len(PERT))).astype('float32'); cand = clip_pert(cand)
        e = evaded(target, cand); out[todo[e]] = cand[e]; ok[todo[e]] = True
    return out
def tree_thresholds(target):
    df = target.get_booster().trees_to_dataframe(); df = df[df.Feature != 'Leaf']; th = {}
    names = {f: i for i, f in enumerate(FEATS)}
    for f, g in df.groupby('Feature'): th[names[f] if f in names else int(f[1:])] = np.unique(g.Split.values.astype('float32'))
    return th
def gain_rank(target):
    names = {f: i for i, f in enumerate(FEATS)}; sc = target.get_booster().get_score(importance_type='total_gain'); out = []
    for k, v in sc.items():
        i = names[k] if k in names else int(k[1:])
        if MASK[i]: out.append((v, i))
    got = [i for v, i in sorted(out, reverse=True)]; return got + [i for i in PERT if i not in got]      # features the model never splits on are still candidates
def atk_whitebox(target, rows, eps, k_cand=K_CAND, iters=WB_ITERS):
    th = tree_thresholds(target); cand_feats = gain_rank(target)[:k_cand]; x0 = Xv[rows].copy()
    lo = np.maximum(x0 - eps * sd1, LO); hi = np.minimum(x0 + eps * sd1, HI)                          # budget ball intersected with the training range
    x = x0.copy(); pa = proba(target, x)[:, 2]; done = proba(target, x).argmax(1) != 2
    for it in range(iters):
        act = np.flatnonzero(~done)
        if not len(act): break
        C = []
        for f in cand_feats:
            t = th.get(f, np.array([], 'float32')); v = x[act, f]
            j = np.searchsorted(t, v, side='right'); up = np.where(j < len(t), t[np.minimum(j, len(t) - 1)] if len(t) else v, v)
            j2 = np.searchsorted(t, v, side='left') - 1; dn = np.where(j2 >= 0, np.nextafter(t[np.maximum(j2, 0)], np.float32(-np.inf)) if len(t) else v, v)
            for val in (lo[act, f], hi[act, f], np.clip(up, lo[act, f], hi[act, f]), np.clip(dn, lo[act, f], hi[act, f])): C.append((f, val.astype('float32')))
        n = len(C); big = np.repeat(x[act][:, None, :], n, axis=1)
        for c, (f, val) in enumerate(C): big[:, c, f] = val
        P = proba(target, big.reshape(-1, F))[:, 2].reshape(len(act), n); best = P.argmin(1); bp = P[np.arange(len(act)), best]; improve = bp < pa[act] - 1e-9
        for r_i in np.flatnonzero(improve):
            f, val = C[best[r_i]]; x[act[r_i], f] = val[r_i]
        pa[act[improve]] = bp[improve]; done[act[~improve]] = True; done[act] |= (proba(target, x[act]).argmax(1) != 2)
    return x
import gymnasium as gym
from gymnasium import spaces
class EvadeEnv(gym.Env):
    def __init__(self, target, pool, eps, seed):
        super().__init__(); self.pool = pool; self.eps = eps; self.rng = np.random.RandomState(seed); self.booster = target.get_booster()
        self.action_space = spaces.Box(-1, 1, (len(PERT),), np.float32); self.observation_space = spaces.Box(-10, 10, (F + 3 + 1,), np.float32)
    def _q(self, x): return int(self.booster.inplace_predict(x[None, :].astype(np.float32)).argmax())
    def _obs(self): return np.concatenate([np.clip(to_z(self.x), -10, 10), np.eye(3)[self.lab], [self.k / PPO_MAX_STEPS]]).astype(np.float32)
    def reset(self, seed=None, options=None):
        self.x0 = Xv[self.rng.choice(self.pool)].copy(); self.x = self.x0.copy(); self.k = 0; self.lab = self._q(self.x); return self._obs(), {}
    def step(self, a):
        d = np.clip((self.x - self.x0)[PERT] / sd1[PERT] + a * self.eps, -self.eps, self.eps); self.x[PERT] = np.clip(self.x0[PERT] + d * sd1[PERT], LO[PERT], HI[PERT])
        self.k += 1; self.lab = self._q(self.x); ev_ = self.lab != 2; r = (10.0 if ev_ else -1.0) - 0.01 * float((a ** 2).sum())
        return self._obs(), r, bool(ev_), bool(self.k >= PPO_MAX_STEPS and not ev_), {}
def atk_ppo(target, rows, eps, seed):
    from sb3_contrib import RecurrentPPO
    pool = tr[y3[tr] == 2]; env = EvadeEnv(target, pool, eps, seed)                                   # trains on attack windows of the training runs only
    model = RecurrentPPO('MlpLstmPolicy', env, n_steps=128, batch_size=64, learning_rate=3e-4, gamma=0.99, seed=seed, verbose=0, policy_kwargs=dict(lstm_hidden_size=64, n_lstm_layers=1), device='cpu').learn(PPO_TIMESTEPS)
    x0 = Xv[rows]; x = x0.copy(); ok = evaded(target, x0); state = None; start_ = np.ones(len(rows), bool)
    for k in range(PPO_MAX_STEPS):
        lab = proba(target, x).argmax(1); obs = np.concatenate([np.clip(to_z(x), -10, 10), np.eye(3)[lab], np.full((len(rows), 1), k / PPO_MAX_STEPS)], 1).astype(np.float32)
        a, state = model.predict(obs, state=state, episode_start=start_, deterministic=True); start_ = np.zeros(len(rows), bool)
        d = np.clip((x - x0)[:, PERT] / sd1[PERT] + a * eps, -eps, eps); xn = x.copy(); xn[:, PERT] = np.clip(x0[:, PERT] + d * sd1[PERT], LO[PERT], HI[PERT])
        new = (proba(target, xn).argmax(1) != 2); x = np.where(ok[:, None], x, xn); ok |= new
    return x


In [ ]:
def disp_ok(x, rows):
    d = np.abs(x - Xv[rows]) / sd1; assert d[:, ~MASK].max() == 0, 'non-controllable feature changed'; return float(d.max())
RES = {}; ROWSREC = {}; INFO = {}; t0 = time.time()
for s in SEEDS:
    target = fit_target(s); f1c = f1_score(y3[te], target.predict(Xv[te]), average='macro', labels=[0, 1, 2]); clean_miss = float((target.predict(Xv[EV]) != 2).mean()); net, agree = train_surrogate(target, s)
    INFO[s] = dict(clean_macro_f1=f1c, clean_miss=clean_miss, surrogate_agreement=agree); print(f'seed {s}: clean macro-F1 {f1c:.4f}, clean miss on held-out attack windows {clean_miss:.4f}, surrogate agreement {agree:.3f} ({time.time()-t0:.0f}s)')
    for eps in EPS_LIST:
        for name, fn in (('PGD_grey', lambda: atk_pgd(net, EV, eps)), ('Random', lambda: atk_random(target, EV, eps, s)), ('WhiteBox', lambda: atk_whitebox(target, EV, eps))):
            xa = fn(); assert disp_ok(xa, EV) <= eps + 1e-3, name; e = evaded(target, xa); RES[(s, name, eps)] = float(e.mean()); ROWSREC[(s, name, eps)] = e
            print(f'   {name:9s} eps={eps}: evasion {e.mean():.4f}')
    for eps in PPO_EPS:
        xa = atk_ppo(target, EV, eps, s); assert disp_ok(xa, EV) <= eps + 1e-3; e = evaded(target, xa); RES[(s, 'PPO_budgeted', eps)] = float(e.mean()); ROWSREC[(s, 'PPO_budgeted', eps)] = e; print(f'   PPO_budgeted eps={eps}: evasion {e.mean():.4f} ({time.time()-t0:.0f}s)')
    ROWSREC[(s, 'clean', 0.0)] = (target.predict(Xv[EV]) != 2)


In [ ]:
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0
    for rank, i in enumerate(o): run = max(run, (len(p) - rank) * p[i]); adj[i] = min(1.0, run)
    return adj
R = pd.DataFrame([dict(seed=s, attacker=a, eps=e, evasion=v) for (s, a, e), v in RES.items()]); tab = R.groupby(['attacker', 'eps']).evasion.agg(['mean', 'std']).round(4); print(tab.to_string())
rk = run_key[EV]; uruns = np.unique(rk); grp = {r_: np.flatnonzero(rk == r_) for r_ in uruns}
def burst_ci(keys, seed_boot=0):
    M = np.mean([ROWSREC[k].astype(float) for k in keys], 0); rb = np.random.RandomState(seed_boot); out = []
    for _ in range(N_BOOT):
        pick_ = rb.choice(uruns, len(uruns), replace=True); out.append(np.concatenate([M[grp[r_]] for r_ in pick_]).mean())
    return float(M.mean()), float(np.percentile(out, 2.5)), float(np.percentile(out, 97.5))
CI = {(a, e): burst_ci([(s, a, e) for s in SEEDS]) for (a, e) in sorted({(a, e) for (_, a, e) in RES}, key=lambda t: (t[1], t[0]))}; CI[('clean', 0.0)] = burst_ci([(s, 'clean', 0.0) for s in SEEDS])
cit = pd.DataFrame([dict(attacker=a, eps=e, evasion=v[0], ci_lo=v[1], ci_hi=v[2]) for (a, e), v in CI.items()]); print('\n95% CI, bootstrap over runs:'); print(cit.round(4).to_string(index=False))
pairs = []
for e in EPS_LIST:
    piv = R[R.eps == e].pivot(index='seed', columns='attacker', values='evasion')
    fam = [('WhiteBox', 'Random'), ('WhiteBox', 'PGD_grey'), ('Random', 'PGD_grey')] + ([('PPO_budgeted', 'Random'), ('PPO_budgeted', 'PGD_grey'), ('WhiteBox', 'PPO_budgeted')] if e in PPO_EPS else []); ps = []; start_i = len(pairs)
    for a, b in fam:
        p = stats.ttest_rel(piv[a], piv[b])[1] if len(SEEDS) > 1 else np.nan; ps.append(p); pairs.append(dict(eps=e, a=a, b=b, diff=(piv[a] - piv[b]).mean(), p_raw=p))
    for i_, h_ in zip(range(start_i, len(pairs)), holm(np.nan_to_num(ps, nan=1.0))): pairs[i_]['p_holm'] = h_
PT = pd.DataFrame(pairs); print('\npaired t-tests over seeds (Holm within budget; model variance only):'); print(PT.round(4).to_string(index=False))


In [ ]:
vk = D.variant.values[EV]
PV = pd.DataFrame({a: pd.Series(np.mean([ROWSREC[(s, a, 0.1)] for s in SEEDS], 0)).groupby(vk).mean() for a in ['PGD_grey', 'Random', 'WhiteBox'] + (['PPO_budgeted'] if 0.1 in PPO_EPS else [])})
PV['clean'] = pd.Series(np.mean([ROWSREC[(s, 'clean', 0.0)] for s in SEEDS], 0)).groupby(vk).mean(); print('\nper-variant evasion at 0.1 sigma (mean over seeds):'); print(PV.round(3).to_string())
tgt = fit_target(SEEDS[0]); SENS = {}
for kc in (3, 6, len(PERT)): SENS[kc] = float(evaded(tgt, atk_whitebox(tgt, EV, 0.1, k_cand=kc)).mean()); print(f'white-box 0.1 sigma, K_CAND={kc}: {SENS[kc]:.4f}')


In [ ]:
T = {'attacks': R, 'summary': tab.reset_index(), 'burst_ci': cit, 'paired': PT, 'per_variant': PV.reset_index()}
for k_, t_ in T.items(): t_.to_csv(f'cpps_adv_{k_}_{PIPELINE_VERSION}.csv', index=False)
joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'EPS_LIST': EPS_LIST, 'PPO_EPS': PPO_EPS, 'tables': T, 'burst_ci': CI, 'seed_info': INFO, 'n_pert': len(PERT), 'n_eval': len(EV), 'whitebox_kcand_check': SENS, 'controllable': [FEATS[i] for i in PERT]}, f'cpps_adv_results_{PIPELINE_VERSION}.pkl')
print('saved', [f for f in os.listdir('.') if f.startswith('cpps_adv')])
